Building the PyPSA model that minimises total annual system costs

In [ ]:


import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from paths import RAW, PROCESSED, RESULTS, savefig

import pypsa
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
# Regions produced by 01_regions_centroids.ipynb
regions_m = gpd.read_file(PROCESSED / "gadm_aut_level1.geojson").to_crs("EPSG:3035").set_index("NAME_1")

n = pypsa.Network()
# 6-hourly (not 3-hourly): the 3h/2920-snapshot LP was too large for HiGHS to solve to
# optimality even with interior-point + rescaling + noise cleanup within a reasonable time
# budget (IPM was genuinely converging, just too slowly for a ~660k-row problem). Halving
# snapshot count to 1460 roughly halves the size of - and cost of factorizing - that system.
n.set_snapshots(pd.date_range("2019-01-01", "2019-12-31 23:00", freq="6h"))

# set_snapshots does NOT infer weighting from the frequency - each snapshot defaults to
# weight 1, which would silently treat every 6-hour step as 1 hour (undercounting annual
# energy/cost/CO2 by 6x). Set it explicitly so dispatch * weighting = correct MWh per step.
n.snapshot_weightings.loc[:, :] = 6

Adding Buses (Buses represent regions)



In [19]:
region_centroids = gpd.read_file(PROCESSED / "austria_region_centroids.geojson")
region_centroids = region_centroids.to_crs("EPSG:3035")
region_centroids = region_centroids.set_index("region")

for region in regions_m.index:
    n.add(
        "Bus",
        region,
        x=region_centroids.loc[region, "x"], #locations of centroids are added
        y=region_centroids.loc[region, "y"],
    )

Adding Loads

In [ ]:
regional_load = pd.read_csv(PROCESSED / "regional_load_AUT.csv", index_col=0, parse_dates=True)

# regional_load_AUT.csv is timestamped for 2013 (the year the underlying gegis/load.csv
# covers), but the network's snapshots are 2019 (matching the ERA5 weather cutout).
# reindex() on mismatched calendar dates would silently produce all-NaN load, which PyPSA
# treats as no demand at all - align by hour-of-year position instead (both years are
# non-leap, 365 days / 8760 hours, so a 1:1 positional remap is exact).
regional_load_6h = regional_load.resample("6h").mean()
assert len(regional_load_6h) == len(n.snapshots)
regional_load_6h.index = n.snapshots

for region in regions_m.index:
    n.add(
        "Load",
        f"load {region}",
        bus=region,
        p_set=regional_load_6h[region],
    )

 Technology Assumptions (costs, efficiencies, lifetimes, etc.)

In [ ]:
cost_year = 2030 #could be another year too
costs_url = f"https://raw.githubusercontent.com/PyPSA/technology-data/master/outputs/costs_{cost_year}.csv"
costs = pd.read_csv(costs_url, index_col=[0, 1])

discount_rate = 0.07 #given

# All capital_cost/marginal_cost values below are divided by COST_SCALE (working in k€
# instead of €). HiGHS's own solver log flagged the LP's cost coefficients spanning
# [5, 3e5] - a 60,000x range - as a likely cause of the solver failing to converge.
# Every cost is rescaled consistently, so relative costs (and thus the optimal solution)
# are unaffected; only the reported objective needs multiplying back by COST_SCALE.
COST_SCALE = 1000

def annuity(r, lifetime):
    return r / (1 - (1 + r) ** (-lifetime))

r
def get_cost(tech, parameter, default=0.0):
    try:
        return costs.loc[(tech, parameter), "value"]
    except KeyError:
        return default

power_techs = ["CCGT", "OCGT", "coal", "oil", "biomass", "hydro", "onwind", "solar-utility"]

fuel_row = {
    "CCGT": "gas", "OCGT": "gas", "coal": "coal", "oil": "oil", "biomass": "biomass",
    "hydro": None, "onwind": None, "solar-utility": None,
}

cost_table = pd.DataFrame(index=power_techs)
cost_table["investment"] = [get_cost(t, "investment") for t in power_techs]       # EUR/kW
cost_table["FOM"] = [get_cost(t, "FOM") for t in power_techs]                     # %/yr
cost_table["VOM"] = [get_cost(t, "VOM") for t in power_techs]                     # EUR/MWh
cost_table["efficiency"] = [get_cost(t, "efficiency", default=1.0) for t in power_techs]
cost_table["lifetime"] = [get_cost(t, "lifetime", default=25) for t in power_techs]
cost_table["fuel_cost"] = [get_cost(fuel_row[t], "fuel") if fuel_row[t] else 0.0 for t in power_techs]
cost_table["co2_intensity"] = [get_cost(fuel_row[t], "CO2 intensity") if fuel_row[t] else 0.0 for t in power_techs]

cost_table["capital_cost"] = (
    (annuity(discount_rate, cost_table["lifetime"]) + cost_table["FOM"] / 100)
    * cost_table["investment"] * 1000 / COST_SCALE  # EUR/kW -> EUR/MW -> k€/MW
)  # k€/MW/yr
cost_table["marginal_cost"] = (
    cost_table["VOM"] + cost_table["fuel_cost"] / cost_table["efficiency"]
) / COST_SCALE  # k€/MWh
cost_table["co2_emissions"] = cost_table["co2_intensity"] / cost_table["efficiency"]  # tCO2/MWh_el, not a cost - unscaled

cost_table.loc["biomass", "co2_emissions"] = 0.0

cost_table


Adding Solar and Onshore Wind Generators

One extendable generator per region for each technology, using the capacity factor time
series from `04_weather_capacity_factors.ipynb` as `p_max_pu` and the annualized costs from
the cost table above.

In [ ]:
# Capacity factor time series per region, resampled to the network's 6-hourly snapshots
wind_cf = xr.open_dataarray(PROCESSED / "wind_capacity_factors_AUT.nc").to_pandas()
solar_cf = xr.open_dataarray(PROCESSED / "solar_capacity_factors_AUT.nc").to_pandas()

wind_cf_6h = wind_cf.resample("6h").mean().reindex(n.snapshots)
solar_cf_6h = solar_cf.resample("6h").mean().reindex(n.snapshots)

# p_max_pu becomes a matrix coefficient (on p_nom) for extendable generators, not just a
# bound - HiGHS's log flagged ~1e-9-scale matrix entries as near-zero noise it had to
# ignore. Floating-point artifacts from the atlite capacity-factor normalization can leave
# values like 1e-10 where the true value should be exactly 0; threshold them away.
wind_cf_6h = wind_cf_6h.where(wind_cf_6h > 1e-6, 0.0)
solar_cf_6h = solar_cf_6h.where(solar_cf_6h > 1e-6, 0.0)

In [ ]:
n.add("Carrier", ["onwind", "solar"], co2_emissions=[0.0, 0.0])

for region in regions_m.index:
    n.add(
        "Generator",
        f"onwind {region}",
        bus=region,
        carrier="onwind",
        p_nom_extendable=True,
        p_max_pu=wind_cf_6h[region],
        capital_cost=cost_table.loc["onwind", "capital_cost"],
        marginal_cost=cost_table.loc["onwind", "marginal_cost"],
    )
    n.add(
        "Generator",
        f"solar {region}",
        bus=region,
        carrier="solar",
        p_nom_extendable=True,
        p_max_pu=solar_cf_6h[region],
        capital_cost=cost_table.loc["solar-utility", "capital_cost"],
        marginal_cost=cost_table.loc["solar-utility", "marginal_cost"],
    )

n.generators

Adding Battery Storage (2h energy-to-power ratio)

Battery cost is split into a power component (inverter) and an energy component (storage
tank), each annualized over its own lifetime, following the standard PyPSA costing approach.
For a given energy-to-power ratio `max_hours`, the annualized cost per MW of power capacity is
`inverter_cost + max_hours * storage_cost`.

Originally added 2h/4h/6h as separate competing StorageUnits so the optimizer could pick
whichever duration mix was cheapest. Cut back to a single 2h duration: three near-substitutable
technologies per region (on top of 9 similar regions and 13 similar transmission links) created
a highly degenerate LP that a 4+ hour HiGHS solve never converged on (primal infeasibility
never shrank, objective diverged) — see the diagnosis discussed before this change. One duration
removes that redundant structure; 2h was chosen as the middle-ground default rather than 4h/6h.

In [ ]:
battery_inverter = costs.loc["battery inverter"]
battery_storage = costs.loc["battery storage"]

# power component (inverter): annualized over its own lifetime -> k€/MW/yr
battery_power_cost = (
    annuity(discount_rate, battery_inverter.loc["lifetime", "value"])
    + battery_inverter.loc["FOM", "value"] / 100
) * battery_inverter.loc["investment", "value"] * 1000 / COST_SCALE  # EUR/kW -> EUR/MW -> k€/MW

# energy component (storage tank): annualized over its own lifetime -> k€/MWh/yr
battery_energy_cost = (
    annuity(discount_rate, battery_storage.loc["lifetime", "value"])
) * battery_storage.loc["investment", "value"] * 1000 / COST_SCALE  # EUR/kWh -> EUR/MWh -> k€/MWh

# round-trip efficiency split symmetrically between charging and discharging
battery_efficiency = battery_inverter.loc["efficiency", "value"] ** 0.5

storage_durations = [2]  # hours - single duration, see markdown above

In [ ]:
n.add("Carrier", "battery", co2_emissions=0.0)

for region in regions_m.index:
    for hours in storage_durations:
        n.add(
            "StorageUnit",
            f"battery {hours}h {region}",
            bus=region,
            carrier="battery",
            p_nom_extendable=True,
            max_hours=hours,
            capital_cost=battery_power_cost + hours * battery_energy_cost,
            efficiency_store=battery_efficiency,
            efficiency_dispatch=battery_efficiency,
            cyclic_state_of_charge=True,
        )

n.storage_units

Adding the Existing Conventional Fleet

One non-extendable generator per (region, technology) from `existing_generators_AUT.csv`
(produced by `05_existing_plant.ipynb`): hydro and gas, already aggregated to a representative
generator per region. Hydro gets its fixed `p_max_pu` (historical generation / capacity ratio,
computed in notebook 05); gas defaults to fully dispatchable (`p_max_pu = 1`).

The power plant database doesn't distinguish CCGT from OCGT for Austria's existing gas fleet,
so it's treated as CCGT here (Austria's existing large gas plants are combined-cycle) — flagged
in case that assumption should change. Capital cost is left at its default of 0 for this
existing, non-extendable fleet: it's a sunk cost that doesn't affect the optimal dispatch/build
decision, only marginal cost does.

In [ ]:
existing_generators = pd.read_csv(PROCESSED / "existing_generators_AUT.csv")

fuel_to_carrier = {"Hydro": "hydro", "Gas": "CCGT"}

n.add(
    "Carrier",
    ["hydro", "CCGT"],
    co2_emissions=cost_table.loc[["hydro", "CCGT"], "co2_emissions"].values,
)

for _, row in existing_generators.iterrows():
    carrier = fuel_to_carrier[row["primary_fuel"]]
    n.add(
        "Generator",
        f"{carrier} existing {row['NAME_1']}",
        bus=row["NAME_1"],
        carrier=carrier,
        p_nom=row["capacity_mw"],
        p_nom_extendable=False,
        p_max_pu=row["p_max_pu"] if pd.notna(row["p_max_pu"]) else 1.0,
        marginal_cost=cost_table.loc[carrier, "marginal_cost"],
    )

n.generators.loc[n.generators.carrier.isin(["hydro", "CCGT"])]

Adding Hydrogen Storage (electrolysis + H2 store + fuel cell)

Not named explicitly in the assignment text seen so far, so this uses standard PyPSA
`technology-data` defaults rather than assignment-specified components. Modeled as three
separate pieces per region, since hydrogen's charge and discharge efficiencies are very
different (unlike the battery's single round-trip figure):

- an electrolyzer (`Link`, electricity → H2, ~62% efficiency)
- an underground H2 store (`Store`, extendable energy capacity — the cheap part per MWh,
  which is the whole point of hydrogen for long-duration/seasonal storage)
- a fuel cell (`Link`, H2 → electricity, ~50% efficiency)

Each requires its own dedicated `Bus` per region to hold the H2 "commodity", separate from
the electricity bus.

In [ ]:
electrolysis = costs.loc["electrolysis"]
fuel_cell = costs.loc["fuel cell"]
h2_store = costs.loc["hydrogen storage underground"]

# electrolyzer: sized by electricity input (MW_e) -> k€/MW/yr
electrolysis_capital_cost = (
    annuity(discount_rate, electrolysis.loc["lifetime", "value"])
    + electrolysis.loc["FOM", "value"] / 100
) * electrolysis.loc["investment", "value"] * 1000 / COST_SCALE  # EUR/kW -> EUR/MW -> k€/MW
electrolysis_efficiency = electrolysis.loc["efficiency", "value"]  # H2 out / elec in

# fuel cell: sized by electricity output (MW_e) -> k€/MW/yr
fuel_cell_capital_cost = (
    annuity(discount_rate, fuel_cell.loc["lifetime", "value"])
    + fuel_cell.loc["FOM", "value"] / 100
) * fuel_cell.loc["investment", "value"] * 1000 / COST_SCALE  # EUR/kW -> EUR/MW -> k€/MW
fuel_cell_efficiency = fuel_cell.loc["efficiency", "value"]  # elec out / H2 in

# underground H2 store: energy capacity -> k€/MWh/yr (very cheap per MWh, unlike the power side)
h2_store_capital_cost = (
    annuity(discount_rate, h2_store.loc["lifetime", "value"])
) * h2_store.loc["investment", "value"] * 1000 / COST_SCALE  # EUR/kWh -> EUR/MWh -> k€/MWh

In [ ]:
n.add("Carrier", ["H2", "H2 electrolysis", "H2 fuel cell", "H2 store"], co2_emissions=0.0)

for region in regions_m.index:
    h2_bus = f"H2 {region}"
    n.add("Bus", h2_bus, carrier="H2")

    n.add(
        "Link",
        f"electrolysis {region}",
        bus0=region,
        bus1=h2_bus,
        carrier="H2 electrolysis",
        p_nom_extendable=True,
        efficiency=electrolysis_efficiency,
        capital_cost=electrolysis_capital_cost,
    )

    n.add(
        "Link",
        f"fuel cell {region}",
        bus0=h2_bus,
        bus1=region,
        carrier="H2 fuel cell",
        p_nom_extendable=True,
        efficiency=fuel_cell_efficiency,
        capital_cost=fuel_cell_capital_cost,
    )

    n.add(
        "Store",
        f"H2 store {region}",
        bus=h2_bus,
        carrier="H2 store",
        e_nom_extendable=True,
        e_cyclic=True,
        capital_cost=h2_store_capital_cost,
    )

n.links.loc[n.links.carrier.str.startswith("H2")]

Adding Transmission (bidirectional Links between neighbouring regions)

Per the assignment: a bidirectional `Link` (not `Line`, so Kirchhoff's Voltage Law and
transmission losses are neglected) is offered between every pair of neighbouring regions —
"neighbouring" meaning their GADM polygons actually share a border (`touches` on the region
geometries already loaded as `regions_m`). Line length = 1.5 x the crow-fly (straight-line)
distance between the two regions' representative centroids (`region_centroids`, already in a
projected CRS, so a plain Euclidean distance gives that).

700 EUR/MW/km is treated as an investment cost and annuitized like every other technology here
(assumed 40-year line lifetime — not specified in the assignment, so flagged). No `p_nom_max`
cap is set: nothing artificially limits how much capacity can be built on a link — the only
thing that limits it is the link's own annualized cost, which scales with that link's fixed
length.

In [ ]:
line_lifetime = 40  # years - not specified in the assignment, standard AC/DC transmission assumption
line_cost_per_mw_km = annuity(discount_rate, line_lifetime) * 700 / COST_SCALE  # k€/MW/km/yr

neighbours = []
region_names = list(regions_m.index)
for i, a in enumerate(region_names):
    for b in region_names[i + 1:]:
        if regions_m.loc[a, "geometry"].touches(regions_m.loc[b, "geometry"]):
            crow_fly_km = (
                region_centroids.loc[a, "geometry"].distance(region_centroids.loc[b, "geometry"]) / 1000
            )
            neighbours.append((a, b, 1.5 * crow_fly_km))

neighbours = pd.DataFrame(neighbours, columns=["region_a", "region_b", "length_km"])
neighbours

In [ ]:
n.add("Carrier", "AC", co2_emissions=0.0)

for _, row in neighbours.iterrows():
    n.add(
        "Link",
        f"{row['region_a']} - {row['region_b']}",
        bus0=row["region_a"],
        bus1=row["region_b"],
        carrier="AC",
        p_nom_extendable=True,
        p_min_pu=-1,  # bidirectional
        efficiency=1.0,  # transmission losses neglected
        capital_cost=line_cost_per_mw_km * row["length_km"],
    )

n.links.loc[n.links.carrier == "AC"]

Exporting the Built Network

The two mandatory scenarios (no CO2 limit vs. 100% CO2 reduction) are solved in separate
notebooks, `07_scenario_no_co2_limit.ipynb` and `08_scenario_100pct_co2_reduction.ipynb`,
instead of here. Reasons:

- **Process isolation for solver settings.** Running both scenarios sequentially in one
  process led to a per-scenario `time_limit` safety net not actually bounding the second
  solve independently (its HiGHS log reported ~3900s elapsed against a 900s limit) — solving
  each scenario in its own fresh Python process removes any possibility of that kind of
  solver-state leakage between runs.
- **Speed.** Building this network (~2920 snapshots x 9 regions x every technology added
  above) is the cheap part; solving it is the expensive part. Building once and exporting
  means you don't pay the build cost again every time you want to retune solver settings or
  re-run just one scenario.

The only carrier in this model with non-zero `co2_emissions` is CCGT — everything else
(hydro, onwind, solar, battery, H2, transmission) is already zero-emission.

In [ ]:
# COST_SCALE is needed by the scenario notebooks too, to convert the solved objective
# (in k€) back to true EUR - stash it on the network so it survives the netCDF round-trip.
n.meta["cost_scale"] = COST_SCALE

n.export_to_netcdf(str(PROCESSED / "pypsa_network_base.nc"))